In [19]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [20]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [21]:
df.drop(columns=['id', 'Unnamed: 32'], inplace =True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [22]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

In [23]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [24]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

numpy to tensors

In [25]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

define model

In [26]:
class MySingleNN():
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_predict = torch.sigmoid(z)
        return y_predict
    
    def loss_function(self, y_predict, y):
        # clamp
        epsilon = 1e-7
        y_predict = torch.clamp(y_predict, epsilon, 1- epsilon)
        # calculate loss
        loss = -(y_train_tensor * torch.log(y_predict) + (1 - y_train_tensor) * torch.log(1 - y_predict)).mean()
        return loss

define parameters

In [27]:
learning_rate = 0.1
epochs = 25

training pipeline

In [28]:
# create model
model = MySingleNN(X_train_tensor)

# train model:- 

# define loop
for epoch in range(epochs):
    #1. forward pass
        y_predict = model.forward(X_train_tensor)

    #2. loss calculate
        loss = model.loss_function(y_predict, y_train_tensor)
        
    #3. backward pass, calculate derivatives
        loss.backward()
    
    #4. param update
        with torch.no_grad():
            model.weights -= learning_rate * model.weights.grad
            model.bias -=  learning_rate * model.bias.grad

        model.weights.grad.zero_()
        model.bias.grad.zero_()
    
        print(f"Epoch: {epoch}, and the loss with it: {loss}")   

Epoch: 0, and the loss with it: 4.027460849967977
Epoch: 1, and the loss with it: 3.9194864000471505
Epoch: 2, and the loss with it: 3.8083841734932093
Epoch: 3, and the loss with it: 3.693774165776006
Epoch: 4, and the loss with it: 3.5765467182654977
Epoch: 5, and the loss with it: 3.4525381019431416
Epoch: 6, and the loss with it: 3.319215800769643
Epoch: 7, and the loss with it: 3.1764410373506116
Epoch: 8, and the loss with it: 3.0319900448506854
Epoch: 9, and the loss with it: 2.884738049234737
Epoch: 10, and the loss with it: 2.7279549904271643
Epoch: 11, and the loss with it: 2.566295550348852
Epoch: 12, and the loss with it: 2.4038639764886023
Epoch: 13, and the loss with it: 2.235412773843683
Epoch: 14, and the loss with it: 2.062307050094813
Epoch: 15, and the loss with it: 1.8924917415971791
Epoch: 16, and the loss with it: 1.7205034683899814
Epoch: 17, and the loss with it: 1.552051702503633
Epoch: 18, and the loss with it: 1.3919296125897263
Epoch: 19, and the loss with i

evaluation

In [29]:
with torch.no_grad():
    y_predict = model.forward(X_test_tensor)
    y_predict = (y_predict > 0.9).float()
    accuracy = (y_predict == y_test_tensor).float().mean()
print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6060326099395752
